# MapleMart Customer Campaign Analytics - End-to-End Pipeline

Connection -> validation -> feature engineering -> train/test split -> train -> evaluate -> predict -> store -> (optional) AI reports.

Run `scripts/setup.sh` (or `setup.ps1`) first so the database and containers are up before running this notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `import Configuration...` etc. resolve from Python/

from Database.database import test_connection
from Database.repository import get_customer_analytics_dataset
from DataPreparation.validation import validate
from DataPreparation.feature_engineering import engineer_features
from MachineLearning.training import split_data, train_model
from MachineLearning.evaluation import evaluate
from MachineLearning.prediction import predict_all, store_predictions_and_metrics
from Configuration.config import ML

## 1. Connect

In [ ]:
assert test_connection(), 'Could not connect - check .env / Python/Configuration/settings.json'

## 2. Pull the analytics dataset (vwCustomerAnalytics via uspGeneratePredictionDataset)

In [ ]:
raw = get_customer_analytics_dataset()
raw.head()

## 3. Validate

In [ ]:
report = validate(raw)
report

## 4. Feature engineering (10 required features + target)

In [ ]:
X, y, customer_ids = engineer_features(raw, for_training=True)
X.describe()

## 5. Train/test split

80/20 stratified split (see `Configuration/settings.json` -> `ml.test_size`): stratification keeps the (usually low) positive-response rate proportionally represented in both sets, which matters a lot with an imbalanced target like this one.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

## 6. Train

Algorithm: **Random Forest**. Chosen because it handles the mix of numeric and ordinal-encoded categorical features without scaling, is robust to the class imbalance in `PurchaseCompleted`, and exposes `feature_importances_` for the marketing-interpretation narrative.

In [ ]:
model, train_duration = train_model(X_train, y_train)

## 7. Evaluate

In [ ]:
metrics = evaluate(model, X_test, y_test)
for k, v in metrics['business_interpretation'].items():
    print(f'{k}: {v}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cm = np.array(metrics['confusion_matrix'])
fig, ax = plt.subplots()
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['No', 'Yes'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['No', 'Yes'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.show()

## 8. Predict for the full customer base and store results

Scores every customer (not just the test split) so `CustomerPrediction` covers the whole base for Power BI / AI reporting.

In [ ]:
X_full, all_customer_ids = engineer_features(raw, for_training=False)
predictions = predict_all(model, X_full, all_customer_ids)
predictions.head()

In [ ]:
store_predictions_and_metrics(predictions, metrics, train_duration)

## 9. (Optional) Generate the 5 AI reports

Requires the Ollama container running with the `mistral` model pulled (`./scripts/setup.sh --with-ai`).

In [ ]:
from Ollama.client import is_available
from Ollama.report_generator import generate_all_reports

if is_available():
    generate_all_reports()
else:
    print('Ollama not reachable - skipping AI report generation.')